<center><h1>Vosskuehler_Simon_HW7</h1></center>

Name: Simon Vosskuehler
<br>
Github Username: SimonVoss17
<br>
USC ID: 9916612281

## 1. Multi-class and Multi-Label Classification Using Support Vector Machines

Import packages

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.multiclass import OneVsRestClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.svm import LinearSVC
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import balanced_accuracy_score
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score



### (a) Download the Anuran Calls (MFCCs) Data Set

In [2]:
df = pd.read_csv("Anuran Calls (MFCCs)/Frogs_MFCCs.csv")
df.head()


,MFCCs_ 1,MFCCs_ 2,MFCCs_ 3,MFCCs_ 4,MFCCs_ 5,MFCCs_ 6,MFCCs_ 7,MFCCs_ 8,MFCCs_ 9,MFCCs_10,...,MFCCs_17,MFCCs_18,MFCCs_19,MFCCs_20,MFCCs_21,MFCCs_22,Family,Genus,Species,RecordID
0,1.0,0.152936,-0.105586,0.200722,0.317201,0.260764,0.100945,-0.150063,-0.171128,0.124676,...,-0.108351,-0.077623,-0.009568,0.057684,0.118680,0.014038,Leptodactylidae,Adenomera,AdenomeraAndre,1
1,1.0,0.171534,-0.098975,0.268425,0.338672,0.268353,0.060835,-0.222475,-0.207693,0.170883,...,-0.090974,-0.056510,-0.035303,0.020140,0.082263,0.029056,Leptodactylidae,Adenomera,AdenomeraAndre,1
2,1.0,0.152317,-0.082973,0.287128,0.276014,0.189867,0.008714,-0.242234,-0.219153,0.232538,...,-0.050691,-0.023590,-0.066722,-0.025083,0.099108,0.077162,Leptodactylidae,Adenomera,AdenomeraAndre,1
3,1.0,0.224392,0.118985,0.329432,0.372088,0.361005,0.015501,-0.194347,-0.098181,0.270375,...,-0.136009,-0.177037,-0.130498,-0.054766,-0.018691,0.023954,Leptodactylidae,Adenomera,AdenomeraAndre,1
4,1.0,0.087817,-0.068345,0.306967,0.330923,0.249144,0.006884,-0.265423,-0.172700,0.266434,...,-0.048885,-0.053074,-0.088550,-0.031346,0.108610,0.079244,Leptodactylidae,Adenomera,AdenomeraAndre,1


In [3]:
df.shape

(7195, 26)

In [4]:
y = df[["Family", "Genus", "Species"]]
X = df.iloc[:, :22]
X_train, X_test, y_train, y_test = train_test_split(X, y, train_size=0.7, random_state=552)

### (b) Train a classifier for each label

#### (i) Research

Exact match is when the percentage of predicitons where each label is predicited correctly.

Hamming loss is the percentage of wrongly predicted individual labels. \

Hamming score is 1 - Hamming loss

#### (ii) Train a SVM for each of the labels

In [5]:
parameters = {"estimator__gamma": np.logspace(-2, 1, 7),
        "estimator__C": np.logspace(-1,4, 6)}

In [6]:
# Genus

genus_classifier = GridSearchCV(OneVsRestClassifier(SVC(kernel="rbf")), parameters, cv=10, n_jobs=-1)
genus_classifier.fit(X_train, y_train["Genus"])
print(genus_classifier.best_params_, genus_classifier.best_score_)

{'estimator__C': np.float64(10.0), 'estimator__gamma': np.float64(3.1622776601683795)} 0.9916615387042821


In [7]:
# Species

species_classifier = GridSearchCV(OneVsRestClassifier(SVC(kernel="rbf")), parameters, cv=10, n_jobs=-1)
species_classifier.fit(X_train, y_train["Species"])
print(species_classifier.best_params_, species_classifier.best_score_)

{'estimator__C': np.float64(10.0), 'estimator__gamma': np.float64(3.1622776601683795)} 0.9904702735965161


In [8]:
# Family

family_classifier = GridSearchCV(OneVsRestClassifier(SVC(kernel="rbf")), parameters, cv=10, n_jobs=-1)
family_classifier.fit(X_train, y_train["Family"])
print(family_classifier.best_params_, family_classifier.best_score_)


{'estimator__C': np.float64(10.0), 'estimator__gamma': np.float64(3.1622776601683795)} 0.9936440878538295


In [9]:
y_pred = pd.DataFrame({"Family": family_classifier.predict(X_test),
                        "Genus": genus_classifier.predict(X_test),
                        "Species": species_classifier.predict(X_test),
                       })

correct = y_test.values == y_pred.values
print("Exact Match:", correct.all(axis=1).mean())
print("Hamming loss:", 1-correct.mean())

Exact Match: 0.9865678554886521
Hamming loss: 0.01049868766404205


In [10]:

print(balanced_accuracy_score(y_test["Family"], family_classifier.predict(X_test)))
print(balanced_accuracy_score(y_test["Genus"], genus_classifier.predict(X_test)))
print(balanced_accuracy_score(y_test["Species"], species_classifier.predict(X_test)))

0.9710093953738022
0.9647919206365099
0.9682446662214442


98.7% of all frogs had all their labels predicted correctly and about 1% of individual labels were predicted incorrectly

Best C was 10 and best gamma was 3.16 so SVM penalty is 0.1 and width of gaussian kernel is 0.4

Balanced accuracy was 0.971, 0.965, and 0.968 for Family, Genus, and Species

#### (iii) Repeat 1(b)ii with L1-penalized SVMs

In [11]:
parameters = {"linearsvc__C": np.logspace(-3, 3, 7)}

In [12]:
family_classifier_l1 = GridSearchCV(
            make_pipeline(StandardScaler(), 
                          LinearSVC(penalty="l1", dual=False, max_iter=10000)), 
                          parameters, cv=10, n_jobs=-1)

family_classifier_l1.fit(X_train, y_train["Family"])
print(family_classifier_l1.best_params_, family_classifier_l1.best_score_)

{'linearsvc__C': np.float64(1.0)} 0.9386443245290165


In [13]:
genus_classifier_l1 = GridSearchCV(
            make_pipeline(StandardScaler(), 
                          LinearSVC(penalty="l1", dual=False, max_iter=10000)), 
                          parameters, cv=10, n_jobs=-1)

genus_classifier_l1.fit(X_train, y_train["Genus"])
print(genus_classifier_l1.best_params_, genus_classifier_l1.best_score_)

{'linearsvc__C': np.float64(10.0)} 0.9539366972766574


In [14]:
species_classifier_l1 = GridSearchCV(
            make_pipeline(StandardScaler(), 
                          LinearSVC(penalty="l1", dual=False, max_iter=10000)), 
                          parameters, cv=10, n_jobs=-1)

species_classifier_l1.fit(X_train, y_train["Species"])
print(species_classifier_l1.best_params_, species_classifier_l1.best_score_)

{'linearsvc__C': np.float64(10.0)} 0.9588978036542649


In [15]:
y_pred_l1 = pd.DataFrame({"Family": family_classifier_l1.predict(X_test),
                          "Genus": genus_classifier_l1.predict(X_test),
                          "Species": species_classifier_l1.predict(X_test)})
correct = y_test.values == y_pred_l1.values
print("Exact match:", correct.all(axis=1).mean())
print("Hamming loss:", 1-correct.mean())

Exact match: 0.9170912459471978
Hamming loss: 0.05264783078585766


In [16]:

print(balanced_accuracy_score(y_test["Family"], family_classifier_l1.predict(X_test)))
print(balanced_accuracy_score(y_test["Genus"], genus_classifier_l1.predict(X_test)))
print(balanced_accuracy_score(y_test["Species"], species_classifier_l1.predict(X_test)))

0.6927715234006803
0.8314144454977282
0.8912904432450934


Exact match was 0.917 and hamming loss was 0.05

Best C for family was one and 10 for genus and species, with cv accuracies of 0.939, 0.953 and 0.958 respectively. 

So SVM penalty for family was 1 and 0.1 for species and genus.

Balanced accuracy was 0.692, 0.831, and 0.892 for Family, Genus, and Species

#### (iv) Repeat 1(b)iii by using SMOTE or any other method for imbalance

In [17]:
# balancing using class weights

parameters = {"linearsvc__C": np.logspace(-3, 3, 7)}

family_classifier_l1_balanced = GridSearchCV(
                make_pipeline(StandardScaler(), 
                          LinearSVC(penalty="l1", dual=False, max_iter=10000, class_weight="balanced")), 
                          parameters, cv=10, n_jobs=-1)

family_classifier_l1_balanced.fit(X_train, y_train["Family"])
print(family_classifier_l1_balanced.best_params_, family_classifier_l1_balanced.best_score_)

/opt/anaconda3/envs/dsci/lib/python3.11/site-packages/sklearn/svm/_base.py:1298: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
/opt/anaconda3/envs/dsci/lib/python3.11/site-packages/sklearn/svm/_base.py:1298: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(


{'linearsvc__C': np.float64(10.0)} 0.9305015147211966


In [18]:
genus_classifier_l1_balanced = GridSearchCV(
            make_pipeline(StandardScaler(), 
                          LinearSVC(penalty="l1", dual=False, max_iter=10000, class_weight="balanced")), 
                          parameters, cv=10, n_jobs=-1)

genus_classifier_l1_balanced.fit(X_train, y_train["Genus"])
print(genus_classifier_l1_balanced.best_params_, genus_classifier_l1_balanced.best_score_)

/opt/anaconda3/envs/dsci/lib/python3.11/site-packages/sklearn/svm/_base.py:1298: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
/opt/anaconda3/envs/dsci/lib/python3.11/site-packages/sklearn/svm/_base.py:1298: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(


{'linearsvc__C': np.float64(100.0)} 0.9471823818990815


In [19]:
species_classifier_l1_balanced = GridSearchCV(
            make_pipeline(StandardScaler(), 
                          LinearSVC(penalty="l1", dual=False, max_iter=10000, class_weight="balanced")), 
                          parameters, cv=10, n_jobs=-1)

species_classifier_l1_balanced.fit(X_train, y_train["Species"])
print(species_classifier_l1_balanced.best_params_, species_classifier_l1_balanced.best_score_)

/opt/anaconda3/envs/dsci/lib/python3.11/site-packages/sklearn/svm/_base.py:1298: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
/opt/anaconda3/envs/dsci/lib/python3.11/site-packages/sklearn/svm/_base.py:1298: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(


{'linearsvc__C': np.float64(10.0)} 0.9590962163526775


In [20]:
y_pred_l1 = pd.DataFrame({"Family": family_classifier_l1_balanced.predict(X_test),
                          "Genus": genus_classifier_l1_balanced.predict(X_test),
                          "Species": species_classifier_l1_balanced.predict(X_test)})
correct = y_test.values == y_pred_l1.values
print("Exact match:", correct.all(axis=1).mean())
print("Hamming loss:", 1-correct.mean())

Exact match: 0.905048633626679
Hamming loss: 0.058360352014821704


In [21]:

print(balanced_accuracy_score(y_test["Family"], family_classifier_l1_balanced.predict(X_test)))
print(balanced_accuracy_score(y_test["Genus"], genus_classifier_l1_balanced.predict(X_test)))
print(balanced_accuracy_score(y_test["Species"], species_classifier_l1_balanced.predict(X_test)))

0.9182189706138429
0.8867074267454718
0.9026232326083775


Exact match was 0.905 and hamming los was 0.058

Best C for family was 10, 100 for genus and 10 for species, with cv accuracy of 0.931, 0.947 and 0.959 respectively.

SVM penalty was 0.1, 0.01, and 0.1 for family genus and species.

Balanced accuracy was 0.918, 0.887, and 0.903 for family, genus, and species.

Gaussian SVM performed better than the models with linear kernels. This could be because the classes may not be well seperated by a linear boundry. Using balanced classes worsened exact match but improved balanced accuracy, especially for family, for the l1 penalized models. 

## 2. K-Means Clustering on a Multi-Class and Multi-Label Data Set

### (a) Use k-means clustering

In [22]:
scores = {}
for k in range(2, 51):
    clusters = KMeans(n_clusters=k, random_state=552).fit_predict(X)
    scores[k] = silhouette_score(X, clusters)
best_k = max(scores, key=scores.get)
print("best k is:", best_k)

best k is: 4


i picked k using sillouette score with k from 2 to 50 which picked k=4


### (b) Determine which family is the majority

In [23]:
clusters = KMeans(n_clusters=best_k, random_state=552).fit_predict(X)
y_pred = pd.DataFrame(index=y.index)
for column in y.columns:
    majority = pd.crosstab(clusters, y[column]).idxmax(axis=1)
    print(majority, "\n")
    y_pred[column] = pd.Series(clusters).map(majority).values

row_0
0    Leptodactylidae
1    Leptodactylidae
2            Hylidae
3            Hylidae
dtype: str 

row_0
0    Adenomera
1    Adenomera
2    Hypsiboas
3    Hypsiboas
dtype: str 

row_0
0            AdenomeraAndre
1    AdenomeraHylaedactylus
2         HypsiboasCordobae
3      HypsiboasCinerascens
dtype: str 



Most common family in each cluster is 1:Leptodactylidae, 2:Leptodactylidae, 3:          Hylidae, 4:Hylidae

Most common genus in each cluster is 1:Adenomera, 2:Adenomera, 3:Hypsiboas, 4:Hypsiboas

Most common species in each cluster is 1:AdenomeraAndre, 2:AdenomeraHylaedactylus, 3:HypsiboasCordobae, 4:HypsiboasCinerascens

### (c) Calculate the average Hamming distance, Hamming score, and Hamming loss

In [24]:
distance = (y.values != y_pred.values).sum(axis=1)

print("Hamming distance:", distance.mean())
print("Hamming loss:", distance.mean() / 3)
print("Hamming score:", 1 - distance.mean() / 3)

Hamming distance: 0.7002084781097985
Hamming loss: 0.23340282603659948
Hamming score: 0.7665971739634005


### Monte Carlo 

In [25]:
distances = []
for i in range(50):
    scores={}
    k_labels = {}
    for k in range(2, 51):
        labels = KMeans(n_clusters=k, random_state=i).fit_predict(X)
        scores[k] = silhouette_score(X, labels)
        k_labels[k] = labels
    best_k = max(scores, key=scores.get)
    clusters = k_labels[best_k]

    y_pred = pd.DataFrame(index=y.index)
    for column in y.columns:
        majority = pd.crosstab(clusters, y[column]).idxmax(axis=1)
        y_pred[column] = pd.Series(clusters).map(majority).values

    distances.append((y.values != y_pred.values).sum(axis=1).mean())
distances = np.array(distances)


In [26]:
print("Hamming distance: mean", distances.mean(), "std", distances.std())
print("Hamming loss: mean", (distances / 3).mean(), "std", (distances / 3).std())
print("Hamming score: mean", (1 - distances / 3).mean(), "std", (1 - distances / 3).std())

Hamming distance: mean 0.7186685198054203 std 0.12959433572039117
Hamming loss: mean 0.23955617326847353 std 0.04319811190679705
Hamming score: mean 0.7604438267315266 std 0.04319811190679705


over 50 runs average hamming distance was 0.719 with std 0.130, hamming loss 0.240 with std 0.043, and hamming score 0.760 with std 0.043

## 3. ISLR 12.6.2

# (a)

1 and two merge at 0.3, 3 and 4 merge at height 0.45, (1,2) and (3,4) merge at 0.8

![Complete linkage dendrogram](12.6.2.a.png)

# (b)
1 and 2 merge at 0.3, (1,2) and 3 merge at 0.4, (1,2,3) and 4 merge at 0.45

![Complete linkage dendrogram](12.6.2.b.png)

## (c)
cutting between 0.45 and 0.8 gives clusters (1,2) and (3,4)

## (d)
cutting beyween 0.4 and 0.45 gives clusters (1, 2, 3) and 4

## (e)

![Complete linkage dendrogram](12.6.2.e.png)